# Credit Card Fraud Detection Using Autoencoders
### Deep Learning (DL) Lab Project
---
**Abstract:**
Credit card fraud is a major problem in digital financial transactions. Traditional fraud detection methods often struggle to identify new or unusual fraud patterns. This project uses a deep learning technique called an Autoencoder to detect fraudulent transactions by learning the patterns of normal transactions.

The model is trained primarily on legitimate transactions and learns to reconstruct their input features. When an unusual transaction is provided, the autoencoder produces a higher reconstruction error. This error is used as an anomaly score to identify potentially fraudulent transactions.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_recall_curve,
    roc_curve, auc, f1_score, precision_score, recall_score, average_precision_score
)

print('TensorFlow version:', tf.__version__)
print('Keras backend:', keras.backend.backend())

## 2. Load Transaction Dataset & Exploratory Data Analysis
The dataset contains 284,807 transactions with 30 numerical features:
- `Time`: seconds elapsed
- `V1` to `V28`: PCA transformed features
- `Amount`: transaction amount
- `Class`: 0 for Legitimate, 1 for Fraudulent

In [ ]:
data_path = '../dataset/creditcard.csv' if os.path.exists('../dataset/creditcard.csv') else 'dataset/creditcard.csv'
df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)

class_counts = df['Class'].value_counts()
print('Class Distribution:\n', class_counts)
fraud_pct = (class_counts[1] / len(df)) * 100
print(f'Fraud Prevalence: {fraud_pct:.4f}%')
df.head()

In [ ]:
# Visualize Class Imbalance
plt.figure(figsize=(7, 4))
sns.countplot(x='Class', data=df, palette=['#10b981', '#ef4444'])
plt.yscale('log')
plt.title('Class Imbalance (Logarithmic Scale)', fontsize=13, fontweight='bold')
plt.xticks([0, 1], ['Legitimate (0)', 'Fraudulent (1)'])
plt.ylabel('Count (Log Scale)')
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.show()

## 3. Data Preprocessing & Scaling
To avoid data leakage, `StandardScaler` is fitted **strictly on normal training transactions**.
The model inputs are the 30 numerical features (`Time`, `V1-V28`, `Amount`).

In [ ]:
feature_cols = [c for c in df.columns if c != 'Class']

normal_df = df[df['Class'] == 0].reset_index(drop=True)
fraud_df = df[df['Class'] == 1].reset_index(drop=True)

# Split normal into train (80%) and holdout (20%)
norm_train, norm_holdout = train_test_split(normal_df, test_size=0.2, random_state=42)
# Split holdout into val (10%) and test (10%)
norm_val, norm_test = train_test_split(norm_holdout, test_size=0.5, random_state=42)
# Split fraud into val (50%) and test (50%)
fraud_val, fraud_test = train_test_split(fraud_df, test_size=0.5, random_state=42)

scaler = StandardScaler()
scaler.fit(norm_train[feature_cols])

X_train = scaler.transform(norm_train[feature_cols])

# Validation set (normal + fraud)
val_df = pd.concat([norm_val, fraud_val]).sample(frac=1, random_state=42)
X_val = scaler.transform(val_df[feature_cols])
y_val = val_df['Class'].values

# Test set (normal + fraud)
test_df = pd.concat([norm_test, fraud_test]).sample(frac=1, random_state=42)
X_test = scaler.transform(test_df[feature_cols])
y_test = test_df['Class'].values

print('X_train shape (Normal only):', X_train.shape)
print('X_val shape:', X_val.shape, '| Frauds:', int(y_val.sum()))
print('X_test shape:', X_test.shape, '| Frauds:', int(y_test.sum()))

## 4. Deep Learning Model: Autoencoder
Architecture specified in project requirements:
- **Input**: 30 numerical features
- **Encoder 1**: 16 units (ReLU)
- **Encoder 2**: 8 units (ReLU)
- **Latent Space**: 4 units (ReLU)
- **Decoder 1**: 8 units (ReLU)
- **Decoder 2**: 16 units (ReLU)
- **Output**: 30 units (Linear)
- **Loss Function**: Mean Squared Error (MSE)
- **Optimizer**: Adam

In [ ]:
input_dim = 30
inputs = layers.Input(shape=(input_dim,), name='transaction_input')

# Encoder
encoded = layers.Dense(16, activation='relu', name='encoder_1')(inputs)
encoded = layers.Dense(8, activation='relu', name='encoder_2')(encoded)
latent = layers.Dense(4, activation='relu', name='latent_space')(encoded)

# Decoder
decoded = layers.Dense(8, activation='relu', name='decoder_1')(latent)
decoded = layers.Dense(16, activation='relu', name='decoder_2')(decoded)
outputs = layers.Dense(input_dim, activation='linear', name='reconstructed_output')(decoded)

autoencoder = keras.Model(inputs=inputs, outputs=outputs, name='Fraud_Autoencoder')
autoencoder.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
autoencoder.summary()

In [ ]:
# Train Autoencoder on Legitimate Transactions
history = autoencoder.fit(
    X_train,
    X_train,
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

### Training & Validation Loss Plot

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='#2563eb', lw=2)
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='#ef4444', lw=2, linestyle='--')
plt.title('Autoencoder Convergence (MSE Loss)', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Reconstruction Error (MSE)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 5. Calculate Reconstruction Error and Select Optimal Threshold
Formula for Reconstruction Error per sample:
\[ E(x) = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2 \]

Decision Rule:
\[ \text{Prediction} = \begin{cases} \text{Suspicious}, & E(x) > T \\ \text{Legitimate}, & E(x) \le T \end{cases} \]

Rather than picking an arbitrary threshold, we search the Precision-Recall curve on the **validation set** to maximize the F1-Score.

In [ ]:
val_reconstructions = autoencoder.predict(X_val, batch_size=1024, verbose=0)
val_errors = np.mean(np.square(X_val - val_reconstructions), axis=1)

precisions, recalls, pr_thresholds = precision_recall_curve(y_val, val_errors)
denom = precisions[:-1] + recalls[:-1]
f1_scores = np.where(denom > 0, 2 * (precisions[:-1] * recalls[:-1]) / (denom + 1e-12), 0.0)
best_idx = int(np.argmax(f1_scores))
threshold = float(pr_thresholds[best_idx])

print(f'Selected Threshold (T): {threshold:.4f}')
print(f'Validation F1-Score:    {f1_scores[best_idx]:.4f}')
print(f'Validation Precision:   {precisions[best_idx]:.4f}')
print(f'Validation Recall:      {recalls[best_idx]:.4f}')

## 6. Model Evaluation on Held-Out Test Set

In [ ]:
test_reconstructions = autoencoder.predict(X_test, batch_size=1024, verbose=0)
test_errors = np.mean(np.square(X_test - test_reconstructions), axis=1)

y_pred = (test_errors > threshold).astype(int)

print('Test Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraudulent']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, test_errors):.4f}')
print(f'PR-AUC Score:  {average_precision_score(y_test, test_errors):.4f}')

### Reconstruction Error Distribution & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Error Distribution
bins = np.linspace(0, 25, 50)
axes[0].hist(test_errors[y_test == 0], bins=bins, alpha=0.6, color='#10b981', label='Legitimate', density=True)
axes[0].hist(test_errors[y_test == 1], bins=bins, alpha=0.6, color='#ef4444', label='Fraudulent', density=True)
axes[0].axvline(threshold, color='black', linestyle='--', lw=2, label=f'Threshold T={threshold:.2f}')
axes[0].set_title('Reconstruction Error Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Reconstruction Error (MSE)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', xticklabels=['Legitimate', 'Fraudulent'], yticklabels=['Legitimate', 'Fraudulent'], ax=axes[1])
axes[1].set_title('Confusion Matrix Heatmap', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('True Class')

plt.tight_layout()
plt.show()

## 7. Sample Transaction Demonstration
Format matches the required expected output:
```
Transaction A
Reconstruction error: 0.002
Legitimate

Transaction B
Reconstruction error: 0.145
Suspicious
```

In [ ]:
# Transaction A: Normal sample
norm_raw = normal_df.iloc[10][feature_cols].values.reshape(1, -1)
norm_scaled = scaler.transform(norm_raw)
norm_recon = autoencoder.predict(norm_scaled, verbose=0)
err_a = float(np.mean(np.square(norm_scaled - norm_recon)))
label_a = 'Suspicious' if err_a > threshold else 'Legitimate'

# Transaction B: Detected Fraud sample
fraud_raw = fraud_df.iloc[1][feature_cols].values.reshape(1, -1)
fraud_scaled = scaler.transform(fraud_raw)
fraud_recon = autoencoder.predict(fraud_scaled, verbose=0)
err_b = float(np.mean(np.square(fraud_scaled - fraud_recon)))
label_b = 'Suspicious' if err_b > threshold else 'Legitimate'

print('=' * 40)
print('Transaction A')
print(f'Reconstruction error: {err_a:.4f}')
print(f'{label_a}')
print('-' * 40)
print('Transaction B')
print(f'Reconstruction error: {err_b:.4f}')
print(f'{label_b}')
print('=' * 40)